In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.metrics import r2_score, mean_absolute_error, median_absolute_error, mean_absolute_percentage_error, mean_squared_error
from sklearn.preprocessing import TargetEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
import optuna

In [32]:
df = pd.read_csv("../data/processed/jakarta_properties_processed_tes.csv")

In [33]:
df.head()

,price_idr,district,bedrooms,bathrooms,garage,land_size_m2,building_size_m2,cluster,pool,mrt,tol,mall,city_Jakarta Barat,city_Jakarta Pusat,city_Jakarta Selatan,city_Jakarta Timur,city_Jakarta Utara,sub_district
0,21.947041,duren sawit,3.0,2.0,4.0,4.663439,4.691348,1,0,0,0,0,0,0,0,1,0,pondok kelapa
1,23.025851,kebayoran lama,6.0,5.0,2.0,5.379897,5.860786,0,0,0,0,0,0,0,1,0,0,pondok indah
2,23.858760,mampang prapatan,12.0,5.0,9.0,6.957497,6.685861,0,0,0,0,0,0,0,1,0,0,kemang
3,21.465203,kebayoran lama,4.0,5.0,1.0,4.158883,5.068904,0,0,0,0,0,0,0,1,0,0,kebayoran lama
4,21.598735,kebayoran lama,5.0,4.0,1.0,4.158883,5.075174,0,0,0,0,0,0,0,1,0,0,kebayoran lama


In [55]:
models_dict = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=500,
        max_depth=8,
        min_samples_split=5,
        min_samples_leaf=20,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        objective='reg:absoluteerror',
        max_depth=10,
        learning_rate=0.05953139963770414,
        n_estimators=2747,
        subsample=0.8337433250519926,
        colsample_bytree=0.7543943678414979,
        gamma=0.032771478039126015,
        min_child_weight=6,
        reg_alpha=1.716855990003528,
        reg_lambda=6.819289477686187,
        random_state=42
    )
}

In [36]:
X = df.drop(columns=["price_idr"])
y = df["price_idr"]

te_district = TargetEncoder(
    smooth=1,
    cv=5,
    target_type='continuous'
    )

te_subdistrict = TargetEncoder(
    smooth=1,
    cv=5,
    target_type='continuous'
    )



X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train['district'] = te_district.fit_transform(X_train[['district']], y_train).flatten()
X_test['district'] = te_district.transform(X_test[['district']]).flatten()

X_train['sub_district'] = te_subdistrict.fit_transform(X_train[['sub_district']], y_train).flatten()
X_test['sub_district'] = te_subdistrict.transform(X_test[['sub_district']]).flatten()

In [59]:
import json


def train_single_model(model_name,model,X_train,y_train,X_test,y_test):
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    y_test_true = np.expm1(y_test)
    y_pred_true = np.expm1(y_pred)
    error_pct = (np.abs(y_pred_true - y_test_true) / y_test_true)

    result = {
        "Model" : model_name,
        "R2 Score":r2_score(y_test,y_pred),
        "MAE (mean)":mean_absolute_error(y_test_true,y_pred_true),
        "MDAE (median)": median_absolute_error(y_test_true,y_pred_true),
        "MAPE": mean_absolute_percentage_error(y_test_true,y_pred_true),
        "Q25": error_pct.quantile(0.25),
        "Q50": error_pct.quantile(0.50),
        "Q75": error_pct.quantile(0.75)
    }
    with open('../models/metrics_model.json', 'w') as f:
        json.dump(result, f, ensure_ascii=False, indent=4)

    # FEATURE IMPORTANCE
    if model_name in ["Decision Tree", "Random Forest", "XGBoost"]:
        feat_imp = pd.DataFrame({
            'feature': X_train.columns,
            'importance': model.feature_importances_
        }).sort_values(by='importance', ascending=False)

        print(feat_imp.head(10))

    return result


In [44]:
def run_models(X_train, y_train, X_test, y_test, models):
    
    all_results = []

    for model_name, model in models.items():
        print(f"\nTraining {model_name} ...")

        result = train_single_model(
            model_name = model_name,
            model = model,
            X_train = X_train,
            y_train = y_train,
            X_test = X_test,
            y_test = y_test
        )

        all_results.append(result)

        print(f"Metrix Evaluation : {model_name}")
        print(f"R2 Score : {result["R2 Score"]}")
        print(f"MAE (mean): {result["MAE (mean)"]}")
        print(f"MDAE (median) : {result["MDAE (median)"]}")
        print(f"MAPE : {result["MAPE"]}")
        
        all_results_df = pd.DataFrame(all_results)
    return all_results_df

In [39]:
def cross_validation(model, X_train, y_train, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    y_binned = pd.qcut(y_train, q=10, labels=False)
    for model_name, model in models_dict.items():
        print(f"\nCross Validation {model_name} ...")
        scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=skf.split(X_train, y_binned),
            scoring='r2'
        )

        print(f"R2 per fold : {scores}")
        print(f"Rata-rata R2: {scores.mean():.4f}")
        print(f"Std deviasi  : {scores.std():.4f}")

In [40]:
cross_validation(model=models_dict, X_train=X_train, y_train=y_train, n_splits=5)


Cross Validation Linear Regression ...
R2 per fold : [0.87844689 0.87052376 0.86926306 0.86909712 0.87721855]
Rata-rata R2: 0.8729
Std deviasi  : 0.0041

Cross Validation Decision Tree ...
R2 per fold : [0.8786356  0.87079038 0.87059722 0.87254562 0.87566341]
Rata-rata R2: 0.8736
Std deviasi  : 0.0031

Cross Validation Random Forest ...
R2 per fold : [0.88812683 0.8806891  0.8808548  0.8813107  0.88669921]
Rata-rata R2: 0.8835
Std deviasi  : 0.0032

Cross Validation XGBoost ...
R2 per fold : [0.92283154 0.91815571 0.91427113 0.9152385  0.91969656]
Rata-rata R2: 0.9180
Std deviasi  : 0.0031


In [54]:
all_models_result = run_models(X_train = X_train, y_train = y_train, X_test = X_test, y_test = y_test, models = models_dict)
all_models_result


Training Linear Regression ...
Metrix Evaluation : Linear Regression
R2 Score : 0.8737819198849684
MAE (mean): 0.25195028225673755
MDAE (median) : 0.1978030447698913
MAPE : 0.011105622527010977

Training Decision Tree ...
               feature  importance
4         land_size_m2    0.664049
16        sub_district    0.165943
5     building_size_m2    0.113700
0             district    0.024204
2            bathrooms    0.007311
14  city_Jakarta Timur    0.005796
3               garage    0.005201
1             bedrooms    0.004804
15  city_Jakarta Utara    0.003069
12  city_Jakarta Pusat    0.001862
Metrix Evaluation : Decision Tree
R2 Score : 0.8866473290280551
MAE (mean): 0.2131366866670021
MDAE (median) : 0.13608996248795435
MAPE : 0.009416205146182804

Training Random Forest ...
                 feature  importance
4           land_size_m2    0.340003
5       building_size_m2    0.229892
16          sub_district    0.172927
0               district    0.123717
3                 ga

,Model,R2 Score,MAE (mean),MDAE (median),MAPE,Q25,Q50,Q75
0,Linear Regression,0.873782,0.251950,0.197803,0.011106,0.004109,0.008791,0.015479
1,Decision Tree,0.886647,0.213137,0.136090,0.009416,0.002344,0.006037,0.013047
2,Random Forest,0.884012,0.240862,0.189674,0.010611,0.003840,0.008439,0.014960
3,XGBoost,0.922543,0.187788,0.138788,0.008286,0.002723,0.006156,0.011293


In [24]:
cv =KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Objective Function
def objective(trial):

    params = {
        "objective": "reg:absoluteerror",
        "max_depth": trial.suggest_int("max_depth", 3, 10),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            1e-3,
            0.3,
            log=True
        ),

        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            4000
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.5,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.5,
            1.0
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0,
            5
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            10
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            0,
            5
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            0,
            10
        ),

        "random_state": 42,
    }

    model = XGBRegressor(**params)

    score = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    ).mean()

    return score

In [25]:
study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=200
)

print("Best Score :", study.best_value)
print("Best Params:", study.best_params)

[I 2026-07-13 22:06:54,026] A new study created in memory with name: no-name-839f15ea-2fbd-4269-9525-178ba6f64558
[I 2026-07-13 22:07:40,805] Trial 0 finished with value: -0.3016535064210394 and parameters: {'max_depth': 3, 'learning_rate': 0.0791382596368106, 'n_estimators': 3751, 'subsample': 0.6146888036605882, 'colsample_bytree': 0.7500041380526439, 'gamma': 1.2637469019984953, 'min_child_weight': 1, 'reg_alpha': 4.966119416007219, 'reg_lambda': 0.0856875804572399}. Best is trial 0 with value: -0.3016535064210394.
[I 2026-07-13 22:07:59,342] Trial 1 finished with value: -0.31389655010340367 and parameters: {'max_depth': 3, 'learning_rate': 0.03055562712053071, 'n_estimators': 1502, 'subsample': 0.546456552247822, 'colsample_bytree': 0.5903367385070512, 'gamma': 3.36070785216789, 'min_child_weight': 5, 'reg_alpha': 4.409188570401527, 'reg_lambda': 3.714906793148376}. Best is trial 0 with value: -0.3016535064210394.
[I 2026-07-13 22:08:21,238] Trial 2 finished with value: -0.43112185

Best Score : -0.27511947927427527
Best Params: {'max_depth': 10, 'learning_rate': 0.05953139963770414, 'n_estimators': 2747, 'subsample': 0.8337433250519926, 'colsample_bytree': 0.7543943678414979, 'gamma': 0.032771478039126015, 'min_child_weight': 6, 'reg_alpha': 1.716855990003528, 'reg_lambda': 6.819289477686187}


best param = {'max_depth': 10, 'learning_rate': 0.05953139963770414, 'n_estimators': 2747, 'subsample': 0.8337433250519926, 'colsample_bytree': 0.7543943678414979, 'gamma': 0.032771478039126015, 'min_child_weight': 6, 'reg_alpha': 1.716855990003528, 'reg_lambda': 6.819289477686187}


In [60]:
train_single_model(
    model_name = "XGBoost",
    model = models_dict["XGBoost"],
    X_train = X_train,
    y_train = y_train,
    X_test = X_test,
    y_test = y_test
)

                 feature  importance
4           land_size_m2    0.105037
14    city_Jakarta Timur    0.083150
5       building_size_m2    0.075462
15    city_Jakarta Utara    0.064763
16          sub_district    0.061171
11    city_Jakarta Barat    0.058092
13  city_Jakarta Selatan    0.058018
12    city_Jakarta Pusat    0.054475
0               district    0.053248
9                    tol    0.051217


{'Model': 'XGBoost',
 'R2 Score': 0.9256936062627593,
 'MAE (mean)': 1959068073.4933982,
 'MDAE (median)': 654203647.9999933,
 'MAPE': 0.182980686090712,
 'Q25': np.float64(0.04635522535816529),
 'Q50': np.float64(0.12054813257142666),
 'Q75': np.float64(0.2373373264030097)}